# Perform single-cell level quality control

In [1]:
import os
import pathlib
import sys

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

## Load in each single-cell level profile per patient and process

1. Load in the single-cell data (add `patient_id` column).
2. Load in respective organoid qc data (only metadata and cqc columns) to already flag cells that come from a flagged organoid.
   - Also add a flag for if single-cells do not have an organoid segmentation (`parent_organoid` == -1).
   - Also add flag for if the `object_id` for a single-cell is NaN.
3. Concat single-cell data together.

In [3]:
sc_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sc_anno.parquet"
)
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles/organoid_flagged_outliers.parquet"
)

output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

orig_sc_profiles_df = pd.read_parquet(sc_file)
organoid_qc_profiles_df = pd.read_parquet(organoid_file)
# Print the shape and head of the combined organoid profiles DataFrame
print(orig_sc_profiles_df.shape)
orig_sc_profiles_df

(122, 775)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NeighborsCountAdjacent,Nuclei_NoChannel_Neighbors_NeighborsCountByDistance-10,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_ShellsUsed
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,943.928242,572.923715,2.996898,...,0.693960,235.384415,56267.0,218.179218,-3.147345,1,1,1.014637,1,2
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,1129.359747,821.957952,3.015800,...,0.619637,327.336570,127038.0,202.442491,12.589382,0,0,0.941453,1,2
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,861.096578,672.739127,2.988451,...,0.752176,156.878787,53916.0,126.049296,88.982577,1,1,0.586189,1,2
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,937.311452,778.421513,3.026878,...,0.606062,340.764095,116073.0,144.494538,70.537335,0,0,0.671968,1,2
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,854.578128,584.664707,3.207620,...,0.685528,58.806324,19009.0,186.073506,28.958367,2,2,0.865330,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,NF0014_T1,Small Molecule,PI3K and HDAC inhibitor,Investigational,Fimepinostat 1 uM,D5,D5-2,968.957067,787.454343,10.274155,...,0.642339,101.985341,32952.0,139.406121,273.088407,2,2,0.337959,1,4
118,NF0014_T1,Small Molecule,PI3K and HDAC inhibitor,Investigational,Fimepinostat 1 uM,D5,D5-2,810.354301,682.837615,8.328442,...,0.736032,68.561227,28851.0,93.731324,318.763203,0,0,0.227230,0,4
119,NF0014_T1,Small Molecule,PI3K and HDAC inhibitor,Investigational,Fimepinostat 1 uM,E5,E5-2,492.825434,1151.671919,4.978187,...,0.527115,328.792853,56823.0,77.282158,115.566309,0,0,0.400740,0,2
120,NF0014_T1,Small Molecule,PI3K and HDAC inhibitor,Investigational,Fimepinostat 1 uM,E5,E5-2,1250.615285,748.476061,5.233596,...,0.495996,392.828653,91660.0,39.444609,0.000000,0,0,1.000000,1,2


In [4]:
sc_profiles_df = orig_sc_profiles_df.copy()
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_ParentOrganoid",
            "Cell_NoChannel_AreaSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = sc_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

sc_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NeighborsCountAdjacent,Nuclei_NoChannel_Neighbors_NeighborsCountByDistance-10,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_ShellsUsed,Metadata_cqc_nan_detected
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,943.928242,572.923715,2.996898,...,235.384415,56267.0,218.179218,-3.147345,1,1,1.014637,1,2,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,1129.359747,821.957952,3.015800,...,327.336570,127038.0,202.442491,12.589382,0,0,0.941453,1,2,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,861.096578,672.739127,2.988451,...,156.878787,53916.0,126.049296,88.982577,1,1,0.586189,1,2,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,937.311452,778.421513,3.026878,...,340.764095,116073.0,144.494538,70.537335,0,0,0.671968,1,2,False
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,854.578128,584.664707,3.207620,...,58.806324,19009.0,186.073506,28.958367,2,2,0.865330,1,2,False


In [6]:
# Path to patient folders


# Default QC flags
sc_profiles_df["Metadata_cqc_organoid_flagged"] = False
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        ["Metadata_Object_ObjectID", "Nuclei_NoChannel_AreaSizeShape_Volume"]
    ]
    .isna()
    .any(axis=1)
)
sc_profiles_df["Metadata_cqc_missing_parent_organoid"] = (
    sc_profiles_df["Metadata_Object_ParentOrganoid"] == -1
)


organoid_flags_df = organoid_qc_profiles_df[
    ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"]
    + [col for col in organoid_qc_profiles_df.columns if col.startswith("Metadata_cqc")]
]

# Get flagged (object_id, image_set) pairs
flagged_pairs = set(
    organoid_flags_df.loc[
        organoid_flags_df.filter(like="cqc").any(axis=1),
        ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"],
    ].itertuples(index=False, name=None)
)

# Flag SC rows where both parent_organoid & image_set match a flagged organoid
sc_profiles_df["Metadata_cqc_organoid_flagged"] = sc_profiles_df.apply(
    lambda row: (
        (row["Metadata_Object_ParentOrganoid"], row["Metadata_Experiment_WellFOV"])
        in flagged_pairs
    ),
    axis=1,
)

print(sc_profiles_df.shape)
sc_profiles_df.head()

(122, 778)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NeighborsCountAdjacent,Nuclei_NoChannel_Neighbors_NeighborsCountByDistance-10,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_ShellsUsed,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,943.928242,572.923715,2.996898,...,218.179218,-3.147345,1,1,1.014637,1,2,False,False,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,1129.359747,821.957952,3.015800,...,202.442491,12.589382,0,0,0.941453,1,2,False,False,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,861.096578,672.739127,2.988451,...,126.049296,88.982577,1,1,0.586189,1,2,False,False,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,937.311452,778.421513,3.026878,...,144.494538,70.537335,0,0,0.671968,1,2,False,False,False
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,854.578128,584.664707,3.207620,...,186.073506,28.958367,2,2,0.865330,1,2,False,False,False


In [7]:
sc_profiles_df["Nuclei_NoChannel_AreaSizeShape_Volume"].describe()

count       122.000000
mean      68844.434426
std       41929.300059
min        1824.000000
25%       37429.500000
50%       74398.500000
75%       95342.750000
max      229348.000000
Name: Nuclei_NoChannel_AreaSizeShape_Volume, dtype: float64

## Detect outlier single-cells using the non-flagged data

We will attempt to detect instances of poor quality segmentations using the nuclei compartment as the base. The conditions we are using are as follows:

1. Abnormally small or large nuclei using `Volume`
2. Abnormally high `mass displacement` in the nuclei for instances of mis-segmentation of background/no longer in-focus

In [8]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in sc_profiles_df.columns if "Metadata" in x]

In [9]:
# Only process the rows that are not flagged
filtered_plate_df = sc_profiles_df[
    ~(
        sc_profiles_df["Metadata_cqc_nan_detected"]
        | sc_profiles_df["Metadata_cqc_organoid_flagged"]
        | sc_profiles_df["Metadata_cqc_missing_parent_organoid"]
    )
]

# --- Find size based nuclei outliers ---
print("Finding small nuclei outliers...")
small_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": -1,  # Detect very small nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_small_nuclei_outlier"] = False
sc_profiles_df.loc[small_nuclei_outliers.index, "Metadata_cqc_small_nuclei_outlier"] = (
    True
)

print("Finding large nuclei outliers...")
large_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": 2,  # Detect very large nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_large_nuclei_outlier"] = False
sc_profiles_df.loc[large_nuclei_outliers.index, "Metadata_cqc_large_nuclei_outlier"] = (
    True
)

# --- Find mass displacement based nuclei outliers ---
print("Finding high mass displacement outliers...")
high_mass_displacement_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_DNA_Intensity_MassDisplacement": 2,  # Detect high mass displacement
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_mass_displacement_outlier"] = False
sc_profiles_df.loc[
    high_mass_displacement_outliers.index, "Metadata_cqc_mass_displacement_outlier"
] = True

# Print number of outliers (only in filtered rows)
small_count = filtered_plate_df.index.intersection(small_nuclei_outliers.index).shape[0]
large_count = filtered_plate_df.index.intersection(large_nuclei_outliers.index).shape[0]
high_mass_count = filtered_plate_df.index.intersection(
    high_mass_displacement_outliers.index
).shape[0]

print(f"Small nuclei outliers found: {small_count}")
print(f"Large nuclei outliers found: {large_count}")
print(f"High mass displacement outliers found: {high_mass_count}")

# Save updated plate_df with flag columns included
output_file_path = pathlib.Path(f"{output_dir}/sc_flagged_outliers.parquet").resolve()
sc_profiles_df.to_parquet(output_file_path, index=False)

Finding small nuclei outliers...
Number of outliers: 13 (19.40%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 1824.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 23030.0
Finding large nuclei outliers...
Number of outliers: 2 (2.99%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 203127.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 229348.0
Finding high mass displacement outliers...
Number of outliers: 5 (7.46%)
Outliers Range:
Nuclei_DNA_Intensity_MassDisplacement Min: 7.7649455
Nuclei_DNA_Intensity_MassDisplacement Max: 14.5447855
Small nuclei outliers found: 13
Large nuclei outliers found: 2
High mass displacement outliers found: 5


In [10]:
sc_profiles_df.head()

,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_NoChannel_Neighbors_NeighborsCountByDistance-10,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_ShellsUsed,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,943.928242,572.923715,2.996898,...,1,1.014637,1,2,False,False,False,False,False,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,1129.359747,821.957952,3.015800,...,0,0.941453,1,2,False,False,False,False,False,True
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,861.096578,672.739127,2.988451,...,1,0.586189,1,2,False,False,False,False,False,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,937.311452,778.421513,3.026878,...,0,0.671968,1,2,False,False,False,False,False,True
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-2,854.578128,584.664707,3.207620,...,2,0.865330,1,2,False,False,False,True,False,True
